# 206 — TCGA Tumor-Program Robustness

## Objective

Evaluate the robustness of the exploratory TCGA epigenetic-transcriptomic
candidate programs identified in notebook 205.

The analysis will assess:

- candidate-program stability;
- lineage-aware cross-omic consistency;
- sensitivity to major biological and technical confounders;
- dependence on individual tumor lineages;
- robustness of the retained RNA–methylation associations.

This notebook treats the program-discovery artifacts published by notebook 205
as authoritative inputs. It does not repeat RNA-seq, methylation, cohort,
confounder, or program-discovery quality control.

The analysis does not establish biological causality, clinical relevance, or
definitive cross-cancer recurrence. Its purpose is to determine which candidate
programs remain sufficiently robust for downstream biological characterization
and cross-system comparison.

In [1]:
# ====================================================
# Imports
# ====================================================

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =======================================================
# Load program-discovery metadata and candidate catalog
# =======================================================

PROGRAM_DISCOVERY_DIR = Paths.tumor_programs

PROGRAM_DISCOVERY_METADATA_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_program_discovery_metadata.json"
)

CANDIDATE_CATALOG_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_cross_omic_candidate_pair_catalog.csv"
)

with PROGRAM_DISCOVERY_METADATA_PATH.open("r", encoding="utf-8") as file:
    program_discovery_metadata = json.load(file)

candidate_catalog = pd.read_csv(
    CANDIDATE_CATALOG_PATH
)

In [3]:
# =============================================================================
# Load candidate component scores
# =============================================================================

RNA_CANDIDATE_SCORES_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_rna_ica_candidate_scores.csv"
)

METHYLATION_CANDIDATE_SCORES_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_hm450_ica_candidate_scores.csv"
)

rna_candidate_scores = pd.read_csv(
    RNA_CANDIDATE_SCORES_PATH
)

methylation_candidate_scores = pd.read_csv(
    METHYLATION_CANDIDATE_SCORES_PATH
)

In [4]:
# =============================================================================
# Load candidate component loadings
# =============================================================================

RNA_CANDIDATE_LOADINGS_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_rna_ica_candidate_gene_loadings.csv"
)

METHYLATION_CANDIDATE_LOADINGS_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_hm450_ica_candidate_probe_loadings.csv"
)

rna_candidate_loadings = pd.read_csv(
    RNA_CANDIDATE_LOADINGS_PATH
)

methylation_candidate_loadings = pd.read_csv(
    METHYLATION_CANDIDATE_LOADINGS_PATH
)

In [5]:
# =============================================================================
# Load cross-omic project correlations
# =============================================================================

CROSS_OMIC_PROJECT_CORRELATIONS_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_cross_omic_candidate_project_correlations.csv"
)

cross_omic_project_correlations = pd.read_csv(
    CROSS_OMIC_PROJECT_CORRELATIONS_PATH
)

In [6]:
# =============================================================================
# Load sex-at-birth sensitivity results
# =============================================================================

SEX_SENSITIVITY_CORRELATIONS_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_cross_omic_matched_sex_sensitivity.csv"
)

sex_sensitivity_correlations = pd.read_csv(
    SEX_SENSITIVITY_CORRELATIONS_PATH
)

In [7]:
# =============================================================================
# Load confounder covariates
# =============================================================================

CONFOUNDER_COVARIATES_PATH = (
    Paths.metadata
    / "tcga_primary_tumor_multiomic_confounder_covariates.csv"
)

confounder_covariates = pd.read_csv(
    CONFOUNDER_COVARIATES_PATH
)

In [8]:
# =============================================================================
# Inspect robustness-analysis inputs
# =============================================================================

retained_candidates = candidate_catalog[
    candidate_catalog["sex_sensitivity_status"]
    == "retained_after_sex_sensitivity"
].copy()

print(
    f"Retained candidate pairs: {len(retained_candidates)}\n"
    f"RNA candidate scores: {rna_candidate_scores.shape}\n"
    f"Methylation candidate scores: {methylation_candidate_scores.shape}\n"
    f"Confounder covariates: {confounder_covariates.shape}"
)

Retained candidate pairs: 13
RNA candidate scores: (9965, 15)
Methylation candidate scores: (8345, 17)
Confounder covariates: (9965, 18)


In [9]:
# =============================================================================
# Define retained candidate universe
# =============================================================================

retained_candidate_ids = retained_candidates[
    "candidate_pair"
].tolist()

retained_project_correlations = (
    cross_omic_project_correlations.loc[
        cross_omic_project_correlations[
            "candidate_pair"
        ].isin(retained_candidate_ids)
    ]
    .copy()
)

retained_sex_sensitivity = (
    sex_sensitivity_correlations.loc[
        sex_sensitivity_correlations[
            "candidate_pair"
        ].isin(retained_candidate_ids)
    ]
    .copy()
)

In [10]:
# =============================================================================
# Compute leave-one-project-out cross-omic sensitivity
# =============================================================================

lopo_records = []

for candidate_pair, pair_table in (
    retained_project_correlations
    .groupby("candidate_pair", sort=False)
):
    baseline_correlation = pair_table[
        "project_correlation"
    ].median()

    for held_out_project in pair_table["project_id"]:
        remaining_correlations = pair_table.loc[
            pair_table["project_id"].ne(held_out_project),
            "project_correlation",
        ].dropna()

        lopo_records.append(
            {
                "candidate_pair": candidate_pair,
                "held_out_project": held_out_project,
                "baseline_median_correlation": baseline_correlation,
                "lopo_median_correlation": remaining_correlations.median(),
            }
        )

lopo_results = pd.DataFrame(lopo_records)

In [11]:
# =============================================================================
# Summarize leave-one-project-out sensitivity
# =============================================================================

lopo_results["absolute_shift"] = (
    lopo_results["lopo_median_correlation"]
    - lopo_results["baseline_median_correlation"]
).abs()

lopo_results["direction_preserved"] = (
    np.sign(lopo_results["lopo_median_correlation"])
    == np.sign(lopo_results["baseline_median_correlation"])
)

worst_lopo = lopo_results.loc[
    lopo_results.groupby("candidate_pair")["absolute_shift"].idxmax()
]

lopo_summary = (
    worst_lopo[
        [
            "candidate_pair",
            "held_out_project",
            "baseline_median_correlation",
            "lopo_median_correlation",
            "absolute_shift",
        ]
    ]
    .rename(columns={"held_out_project": "most_influential_project"})
    .merge(
        lopo_results.groupby("candidate_pair")["direction_preserved"]
        .all()
        .rename("direction_preserved_all"),
        on="candidate_pair",
    )
    .sort_values("absolute_shift", ascending=False)
    .reset_index(drop=True)
)

lopo_summary

,candidate_pair,most_influential_project,baseline_median_correlation,lopo_median_correlation,absolute_shift,direction_preserved_all
0,CROSS_OMIC_PAIR_02,TCGA-ACC,0.321344,0.346465,0.025121,True
1,CROSS_OMIC_PAIR_09,TCGA-BLCA,0.107940,0.097733,0.010208,True
2,CROSS_OMIC_PAIR_04,TCGA-ACC,0.133099,0.126276,0.006823,True
3,CROSS_OMIC_PAIR_13,TCGA-BLCA,0.101435,0.107871,0.006436,True
4,CROSS_OMIC_PAIR_10,TCGA-ACC,0.107284,0.113138,0.005853,True
5,CROSS_OMIC_PAIR_05,TCGA-ACC,-0.130233,-0.133862,0.003629,True
6,CROSS_OMIC_PAIR_06,TCGA-ACC,0.112888,0.109919,0.002968,True
7,CROSS_OMIC_PAIR_03,TCGA-ACC,-0.179417,-0.182145,0.002729,True
8,CROSS_OMIC_PAIR_14,TCGA-CHOL,-0.101207,-0.103796,0.002589,True
9,CROSS_OMIC_PAIR_11,TCGA-ACC,-0.105210,-0.107275,0.002064,True


In [12]:
# =============================================================================
# Prepare matched candidate-score table
# =============================================================================

retained_rna_components = (
    retained_candidates["rna_component"]
    .drop_duplicates()
    .tolist()
)

retained_methylation_components = (
    retained_candidates["methylation_component"]
    .drop_duplicates()
    .tolist()
)

matched_candidate_scores = (
    rna_candidate_scores[
        [
            "case_submitter_id",
            "project_id",
            *retained_rna_components,
        ]
    ]
    .merge(
        methylation_candidate_scores[
            [
                "case_submitter_id",
                "project_id",
                *retained_methylation_components,
            ]
        ],
        on=["case_submitter_id", "project_id"],
        how="inner",
        validate="one_to_one",
    )
)

In [13]:
# =============================================================================
# Define stratified-bootstrap settings
# =============================================================================

BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_RANDOM_SEED = 206

bootstrap_rng = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED
)

bootstrap_project_ids = (
    retained_project_correlations["project_id"]
    .drop_duplicates()
    .tolist()
)

bootstrap_project_tables = {
    project_id: matched_candidate_scores.loc[
        matched_candidate_scores["project_id"].eq(project_id)
    ]
    for project_id in bootstrap_project_ids
}

In [14]:
# =============================================================================
# Run stratified bootstrap
# =============================================================================

bootstrap_records = []

for bootstrap_iteration in range(BOOTSTRAP_ITERATIONS):
    project_samples = {
        project_id: project_table.iloc[
            bootstrap_rng.integers(
                0,
                len(project_table),
                size=len(project_table),
            )
        ]
        for project_id, project_table in bootstrap_project_tables.items()
    }

    for candidate in retained_candidates.itertuples(index=False):
        project_correlations = []

        for project_id in bootstrap_project_ids:
            project_sample = project_samples[project_id]

            correlation = project_sample[
                candidate.rna_component
            ].corr(
                project_sample[
                    candidate.methylation_component
                ]
            )

            if pd.notna(correlation):
                project_correlations.append(correlation)

        bootstrap_records.append(
            {
                "candidate_pair": candidate.candidate_pair,
                "bootstrap_iteration": bootstrap_iteration,
                "median_project_correlation": np.median(
                    project_correlations
                ),
            }
        )

bootstrap_results = pd.DataFrame(bootstrap_records)

In [15]:
# =============================================================================
# Summarize stratified-bootstrap stability
# =============================================================================

bootstrap_baseline = (
    retained_project_correlations
    .groupby("candidate_pair")["project_correlation"]
    .median()
)

bootstrap_summary = (
    bootstrap_results
    .groupby("candidate_pair")["median_project_correlation"]
    .agg(
        bootstrap_median="median",
        bootstrap_ci_lower=lambda x: x.quantile(0.025),
        bootstrap_ci_upper=lambda x: x.quantile(0.975),
    )
    .reset_index()
)

bootstrap_summary["baseline_median_correlation"] = (
    bootstrap_summary["candidate_pair"]
    .map(bootstrap_baseline)
)

bootstrap_summary["direction_preservation_fraction"] = (
    bootstrap_results.assign(
        direction_preserved=lambda x:
        np.sign(x["median_project_correlation"])
        == np.sign(x["candidate_pair"].map(bootstrap_baseline))
    )
    .groupby("candidate_pair")["direction_preserved"]
    .mean()
    .reindex(bootstrap_summary["candidate_pair"])
    .to_numpy()
)

bootstrap_summary.sort_values(
    "direction_preservation_fraction"
)

,candidate_pair,bootstrap_median,bootstrap_ci_lower,bootstrap_ci_upper,baseline_median_correlation,direction_preservation_fraction
0,CROSS_OMIC_PAIR_02,0.341246,0.281203,0.403576,0.321344,1.0
1,CROSS_OMIC_PAIR_03,-0.177625,-0.221213,-0.130910,-0.179417,1.0
2,CROSS_OMIC_PAIR_04,0.123137,0.082248,0.163422,0.133099,1.0
3,CROSS_OMIC_PAIR_05,-0.125000,-0.160662,-0.083505,-0.130233,1.0
4,CROSS_OMIC_PAIR_06,0.106342,0.065783,0.140721,0.112888,1.0
5,CROSS_OMIC_PAIR_07,0.103148,0.069266,0.143072,0.110080,1.0
6,CROSS_OMIC_PAIR_08,0.097141,0.063231,0.132338,0.109256,1.0
7,CROSS_OMIC_PAIR_09,0.102390,0.059211,0.143753,0.107940,1.0
8,CROSS_OMIC_PAIR_10,0.102405,0.063589,0.139160,0.107284,1.0
9,CROSS_OMIC_PAIR_11,-0.100257,-0.135238,-0.066061,-0.105210,1.0


In [16]:
# =============================================================================
# Prepare continuous-confounder analysis table
# =============================================================================

PRIMARY_CONTINUOUS_CONFOUNDERS = [
    "absolute_purity",
    "leukocyte_fraction",
    "external_panimmune_proliferation_score",
    "gene_assigned_fraction_of_accounted",
    "missing_beta_fraction",
]

ALTERNATIVE_PURITY_COVARIATE = (
    "consensus_purity_estimate"
)

candidate_confounder_table = (
    matched_candidate_scores
    .merge(
        confounder_covariates[
            [
                "case_submitter_id",
                *PRIMARY_CONTINUOUS_CONFOUNDERS,
                ALTERNATIVE_PURITY_COVARIATE,
            ]
        ],
        on="case_submitter_id",
        how="left",
        validate="one_to_one",
    )
)

In [17]:
# =============================================================================
# Define partial-correlation helper
# =============================================================================

def partial_correlation(data, x_column, y_column, covariate_column):
    complete = data[
        [x_column, y_column, covariate_column]
    ].dropna()

    if len(complete) < 4:
        return np.nan

    covariate_matrix = np.column_stack(
        [
            np.ones(len(complete)),
            complete[covariate_column].to_numpy(),
        ]
    )

    x_residuals = (
        complete[x_column].to_numpy()
        - covariate_matrix
        @ np.linalg.lstsq(
            covariate_matrix,
            complete[x_column].to_numpy(),
            rcond=None,
        )[0]
    )

    y_residuals = (
        complete[y_column].to_numpy()
        - covariate_matrix
        @ np.linalg.lstsq(
            covariate_matrix,
            complete[y_column].to_numpy(),
            rcond=None,
        )[0]
    )

    return np.corrcoef(x_residuals, y_residuals)[0, 1]

In [18]:
# =============================================================================
# Compute within-project confounder-adjusted correlations
# =============================================================================

confounder_records = []

for candidate in retained_candidates.itertuples(index=False):
    for project_id, project_table in candidate_confounder_table.groupby(
        "project_id",
        sort=False,
    ):
        for covariate in PRIMARY_CONTINUOUS_CONFOUNDERS:
            n_complete = project_table[
                [
                    candidate.rna_component,
                    candidate.methylation_component,
                    covariate,
                ]
            ].dropna().shape[0]

            confounder_records.append(
                {
                    "candidate_pair": candidate.candidate_pair,
                    "project_id": project_id,
                    "covariate": covariate,
                    "n_complete": n_complete,
                    "adjusted_correlation": partial_correlation(
                        project_table,
                        candidate.rna_component,
                        candidate.methylation_component,
                        covariate,
                    ),
                }
            )

confounder_adjusted_correlations = pd.DataFrame(
    confounder_records
)

In [19]:
# =============================================================================
# Summarize continuous-confounder sensitivity
# =============================================================================

confounder_summary = (
    confounder_adjusted_correlations
    .groupby(["candidate_pair", "covariate"])
    .agg(
        adjusted_median_correlation=(
            "adjusted_correlation",
            "median",
        ),
        projects_with_estimate=(
            "adjusted_correlation",
            "count",
        ),
    )
    .reset_index()
)

confounder_summary["baseline_median_correlation"] = (
    confounder_summary["candidate_pair"]
    .map(bootstrap_baseline)
)

confounder_summary["absolute_shift"] = (
    confounder_summary["adjusted_median_correlation"]
    - confounder_summary["baseline_median_correlation"]
).abs()

confounder_summary["direction_preserved"] = (
    np.sign(confounder_summary["adjusted_median_correlation"])
    == np.sign(confounder_summary["baseline_median_correlation"])
)

confounder_summary.sort_values(
    "absolute_shift",
    ascending=False,
)

,candidate_pair,covariate,adjusted_median_correlation,projects_with_estimate,baseline_median_correlation,absolute_shift,direction_preserved
13,CROSS_OMIC_PAIR_04,leukocyte_fraction,0.076256,30,0.133099,0.056843,True
3,CROSS_OMIC_PAIR_02,leukocyte_fraction,0.360627,30,0.321344,0.039283,True
10,CROSS_OMIC_PAIR_04,absolute_purity,0.101542,33,0.133099,0.031557,True
38,CROSS_OMIC_PAIR_09,leukocyte_fraction,0.138786,30,0.107940,0.030845,True
12,CROSS_OMIC_PAIR_04,gene_assigned_fraction_of_accounted,0.103098,33,0.133099,0.030001,True
...,...,...,...,...,...,...,...
8,CROSS_OMIC_PAIR_03,leukocyte_fraction,-0.178630,30,-0.179417,0.000787,True
33,CROSS_OMIC_PAIR_08,leukocyte_fraction,0.108659,30,0.109256,0.000598,True
63,CROSS_OMIC_PAIR_14,leukocyte_fraction,-0.101474,30,-0.101207,0.000267,True
34,CROSS_OMIC_PAIR_08,missing_beta_fraction,0.109394,33,0.109256,0.000138,True


In [20]:
# =============================================================================
# Compute alternative-purity sensitivity
# =============================================================================

alternative_purity_records = []

for candidate in retained_candidates.itertuples(index=False):
    for project_id, project_table in candidate_confounder_table.groupby(
        "project_id",
        sort=False,
    ):
        n_complete = project_table[
            [
                candidate.rna_component,
                candidate.methylation_component,
                ALTERNATIVE_PURITY_COVARIATE,
            ]
        ].dropna().shape[0]

        alternative_purity_records.append(
            {
                "candidate_pair": candidate.candidate_pair,
                "project_id": project_id,
                "n_complete": n_complete,
                "adjusted_correlation": partial_correlation(
                    project_table,
                    candidate.rna_component,
                    candidate.methylation_component,
                    ALTERNATIVE_PURITY_COVARIATE,
                ),
            }
        )

alternative_purity_correlations = pd.DataFrame(
    alternative_purity_records
)

In [21]:
# =============================================================================
# Summarize alternative-purity sensitivity
# =============================================================================

alternative_purity_summary = (
    alternative_purity_correlations
    .groupby("candidate_pair")
    .agg(
        consensus_purity_median_correlation=(
            "adjusted_correlation",
            "median",
        ),
        consensus_purity_projects=(
            "adjusted_correlation",
            "count",
        ),
    )
    .reset_index()
)

absolute_purity_summary = (
    confounder_summary.loc[
        confounder_summary["covariate"].eq("absolute_purity"),
        [
            "candidate_pair",
            "adjusted_median_correlation",
            "projects_with_estimate",
        ],
    ]
    .rename(
        columns={
            "adjusted_median_correlation":
                "absolute_purity_median_correlation",
            "projects_with_estimate":
                "absolute_purity_projects",
        }
    )
)

purity_sensitivity_summary = (
    alternative_purity_summary
    .merge(
        absolute_purity_summary,
        on="candidate_pair",
        how="left",
    )
)

purity_sensitivity_summary["baseline_median_correlation"] = (
    purity_sensitivity_summary["candidate_pair"]
    .map(bootstrap_baseline)
)

purity_sensitivity_summary

,candidate_pair,consensus_purity_median_correlation,consensus_purity_projects,absolute_purity_median_correlation,absolute_purity_projects,baseline_median_correlation
0,CROSS_OMIC_PAIR_02,0.427652,21,0.297364,33,0.321344
1,CROSS_OMIC_PAIR_03,-0.175205,21,-0.172493,33,-0.179417
2,CROSS_OMIC_PAIR_04,0.128739,21,0.101542,33,0.133099
3,CROSS_OMIC_PAIR_05,-0.158108,21,-0.124139,33,-0.130233
4,CROSS_OMIC_PAIR_06,0.121217,21,0.094947,33,0.112888
5,CROSS_OMIC_PAIR_07,0.134201,21,0.114485,33,0.110080
6,CROSS_OMIC_PAIR_08,0.129820,21,0.112571,33,0.109256
7,CROSS_OMIC_PAIR_09,0.121998,21,0.115905,33,0.107940
8,CROSS_OMIC_PAIR_10,0.084709,21,0.121324,33,0.107284
9,CROSS_OMIC_PAIR_11,-0.052890,21,-0.103608,33,-0.105210


In [22]:
# =============================================================================
# Match purity sensitivities on common project support
# =============================================================================

absolute_purity_project_correlations = (
    confounder_adjusted_correlations.loc[
        confounder_adjusted_correlations["covariate"].eq(
            "absolute_purity"
        ),
        [
            "candidate_pair",
            "project_id",
            "adjusted_correlation",
        ],
    ]
    .rename(
        columns={
            "adjusted_correlation":
                "absolute_purity_adjusted_correlation"
        }
    )
)

matched_purity_correlations = (
    alternative_purity_correlations[
        [
            "candidate_pair",
            "project_id",
            "adjusted_correlation",
        ]
    ]
    .rename(
        columns={
            "adjusted_correlation":
                "consensus_purity_adjusted_correlation"
        }
    )
    .merge(
        absolute_purity_project_correlations,
        on=["candidate_pair", "project_id"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        retained_project_correlations[
            [
                "candidate_pair",
                "project_id",
                "project_correlation",
            ]
        ],
        on=["candidate_pair", "project_id"],
        how="left",
        validate="one_to_one",
    )
)

In [23]:
# =============================================================================
# Restrict purity comparison to common valid project support
# =============================================================================

matched_purity_complete = (
    matched_purity_correlations
    .dropna(
        subset=[
            "project_correlation",
            "absolute_purity_adjusted_correlation",
            "consensus_purity_adjusted_correlation",
        ]
    )
    .copy()
)

matched_purity_summary = (
    matched_purity_complete
    .groupby("candidate_pair")
    .agg(
        matched_baseline_median=(
            "project_correlation",
            "median",
        ),
        matched_absolute_purity_median=(
            "absolute_purity_adjusted_correlation",
            "median",
        ),
        matched_consensus_purity_median=(
            "consensus_purity_adjusted_correlation",
            "median",
        ),
        matched_projects=(
            "project_id",
            "nunique",
        ),
    )
    .reset_index()
)

matched_purity_summary["absolute_purity_shift"] = (
    matched_purity_summary["matched_absolute_purity_median"]
    - matched_purity_summary["matched_baseline_median"]
)

matched_purity_summary["consensus_purity_shift"] = (
    matched_purity_summary["matched_consensus_purity_median"]
    - matched_purity_summary["matched_baseline_median"]
)

matched_purity_summary

,candidate_pair,matched_baseline_median,matched_absolute_purity_median,matched_consensus_purity_median,matched_projects,absolute_purity_shift,consensus_purity_shift
0,CROSS_OMIC_PAIR_02,0.428025,0.441367,0.431181,20,0.013342,0.003156
1,CROSS_OMIC_PAIR_03,-0.179417,-0.172391,-0.178811,20,0.007026,0.000606
2,CROSS_OMIC_PAIR_04,0.150597,0.126916,0.136375,20,-0.023682,-0.014223
3,CROSS_OMIC_PAIR_05,-0.152338,-0.148110,-0.158275,20,0.004228,-0.005937
4,CROSS_OMIC_PAIR_06,0.124097,0.127344,0.119067,20,0.003248,-0.005029
5,CROSS_OMIC_PAIR_07,0.131358,0.134707,0.133919,20,0.003349,0.002561
6,CROSS_OMIC_PAIR_08,0.122860,0.114167,0.125654,20,-0.008693,0.002794
7,CROSS_OMIC_PAIR_09,0.123614,0.121338,0.118314,20,-0.002275,-0.005300
8,CROSS_OMIC_PAIR_10,0.091882,0.096256,0.091237,20,0.004374,-0.000646
9,CROSS_OMIC_PAIR_11,-0.052238,-0.057098,-0.057639,20,-0.004860,-0.005401


In [24]:
# =============================================================================
# Summarize worst continuous-confounder sensitivity
# =============================================================================

worst_confounder_rows = confounder_summary.loc[
    confounder_summary.groupby("candidate_pair")["absolute_shift"].idxmax()
]

continuous_confounder_robustness = (
    worst_confounder_rows[
        [
            "candidate_pair",
            "covariate",
            "adjusted_median_correlation",
            "absolute_shift",
            "projects_with_estimate",
        ]
    ]
    .rename(
        columns={
            "covariate": "most_influential_covariate",
            "adjusted_median_correlation":
                "worst_adjusted_median_correlation",
            "absolute_shift": "maximum_absolute_shift",
        }
    )
    .merge(
        confounder_summary.groupby("candidate_pair")[
            "direction_preserved"
        ]
        .all()
        .rename("direction_preserved_all"),
        on="candidate_pair",
    )
    .sort_values(
        "maximum_absolute_shift",
        ascending=False,
    )
    .reset_index(drop=True)
)

continuous_confounder_robustness

,candidate_pair,most_influential_covariate,worst_adjusted_median_correlation,maximum_absolute_shift,projects_with_estimate,direction_preserved_all
0,CROSS_OMIC_PAIR_04,leukocyte_fraction,0.076256,0.056843,30,True
1,CROSS_OMIC_PAIR_02,leukocyte_fraction,0.360627,0.039283,30,True
2,CROSS_OMIC_PAIR_09,leukocyte_fraction,0.138786,0.030845,30,True
3,CROSS_OMIC_PAIR_12,external_panimmune_proliferation_score,-0.125312,0.021054,30,True
4,CROSS_OMIC_PAIR_06,absolute_purity,0.094947,0.017940,33,True
5,CROSS_OMIC_PAIR_11,leukocyte_fraction,-0.088135,0.017075,30,True
6,CROSS_OMIC_PAIR_13,external_panimmune_proliferation_score,0.086273,0.015162,30,True
7,CROSS_OMIC_PAIR_10,absolute_purity,0.121324,0.014039,33,True
8,CROSS_OMIC_PAIR_14,external_panimmune_proliferation_score,-0.113922,0.012715,30,True
9,CROSS_OMIC_PAIR_08,external_panimmune_proliferation_score,0.100337,0.008919,30,True


In [25]:
# =============================================================================
# Prepare plate-batch sensitivity table
# =============================================================================

candidate_plate_table = (
    matched_candidate_scores
    .merge(
        confounder_covariates[
            [
                "case_submitter_id",
                "rna_plate",
                "methylation_plate",
            ]
        ],
        on="case_submitter_id",
        how="left",
        validate="one_to_one",
    )
)

In [26]:
# =============================================================================
# Define categorical-batch residualization helper
# =============================================================================

def residualize_categorical(data, value_column, batch_column):
    complete = data[
        [value_column, batch_column]
    ].dropna()

    batch_counts = complete[
        batch_column
    ].value_counts()

    valid_batches = batch_counts[
        batch_counts >= 2
    ].index

    complete = complete.loc[
        complete[batch_column].isin(valid_batches)
    ].copy()

    if complete[batch_column].nunique() < 2:
        return pd.Series(
            np.nan,
            index=data.index,
            dtype=float,
        )

    batch_means = complete.groupby(
        batch_column
    )[value_column].transform("mean")

    residuals = pd.Series(
        np.nan,
        index=data.index,
        dtype=float,
    )

    residuals.loc[complete.index] = (
        complete[value_column] - batch_means
    )

    return residuals

In [27]:
# =============================================================================
# Compute within-project plate-adjusted correlations
# =============================================================================

plate_records = []

for candidate in retained_candidates.itertuples(index=False):
    for project_id, project_table in candidate_plate_table.groupby(
        "project_id",
        sort=False,
    ):
        rna_residuals = residualize_categorical(
            project_table,
            candidate.rna_component,
            "rna_plate",
        )

        methylation_residuals = residualize_categorical(
            project_table,
            candidate.methylation_component,
            "methylation_plate",
        )

        valid_samples = (
            rna_residuals.notna()
            & methylation_residuals.notna()
        )

        matched_table = project_table.loc[
            valid_samples,
            [
                candidate.rna_component,
                candidate.methylation_component,
            ],
        ]

        plate_records.append(
            {
                "candidate_pair": candidate.candidate_pair,
                "project_id": project_id,
                "n_complete": valid_samples.sum(),
                "matched_unadjusted_correlation": (
                    matched_table[candidate.rna_component].corr(
                        matched_table[
                            candidate.methylation_component
                        ]
                    )
                ),
                "plate_adjusted_correlation": (
                    rna_residuals.loc[valid_samples].corr(
                        methylation_residuals.loc[valid_samples]
                    )
                ),
            }
        )

plate_adjusted_correlations = pd.DataFrame(
    plate_records
)

In [28]:
# =============================================================================
# Summarize plate-batch sensitivity
# =============================================================================

plate_summary = (
    plate_adjusted_correlations
    .groupby("candidate_pair")
    .agg(
        matched_unadjusted_median=(
            "matched_unadjusted_correlation",
            "median",
        ),
        plate_adjusted_median=(
            "plate_adjusted_correlation",
            "median",
        ),
        projects_with_estimate=(
            "plate_adjusted_correlation",
            "count",
        ),
    )
    .reset_index()
)

plate_summary["absolute_shift"] = (
    plate_summary["plate_adjusted_median"]
    - plate_summary["matched_unadjusted_median"]
).abs()

plate_summary["direction_preserved"] = (
    np.sign(plate_summary["plate_adjusted_median"])
    == np.sign(plate_summary["matched_unadjusted_median"])
)

plate_summary.sort_values(
    "absolute_shift",
    ascending=False,
)

,candidate_pair,matched_unadjusted_median,plate_adjusted_median,projects_with_estimate,absolute_shift,direction_preserved
11,CROSS_OMIC_PAIR_13,0.136554,0.037280,27,0.099274,True
0,CROSS_OMIC_PAIR_02,0.369043,0.293377,27,0.075666,True
9,CROSS_OMIC_PAIR_11,-0.103146,-0.078703,27,0.024443,True
4,CROSS_OMIC_PAIR_06,0.089718,0.112899,27,0.023180,True
10,CROSS_OMIC_PAIR_12,-0.101116,-0.119151,27,0.018035,True
3,CROSS_OMIC_PAIR_05,-0.137337,-0.154498,27,0.017161,True
12,CROSS_OMIC_PAIR_14,-0.098618,-0.089788,27,0.008830,True
5,CROSS_OMIC_PAIR_07,0.109275,0.117559,27,0.008284,True
6,CROSS_OMIC_PAIR_08,0.107848,0.101072,27,0.006776,True
1,CROSS_OMIC_PAIR_03,-0.163303,-0.169411,27,0.006108,True


In [29]:
# =============================================================================
# Summarize project-level plate sensitivity
# =============================================================================

plate_project_sensitivity = (
    plate_adjusted_correlations
    .dropna(
        subset=[
            "matched_unadjusted_correlation",
            "plate_adjusted_correlation",
        ]
    )
    .copy()
)

plate_project_sensitivity["direction_preserved"] = (
    np.sign(
        plate_project_sensitivity[
            "plate_adjusted_correlation"
        ]
    )
    == np.sign(
        plate_project_sensitivity[
            "matched_unadjusted_correlation"
        ]
    )
)

plate_project_sensitivity["absolute_shift"] = (
    plate_project_sensitivity[
        "plate_adjusted_correlation"
    ]
    - plate_project_sensitivity[
        "matched_unadjusted_correlation"
    ]
).abs()

plate_project_summary = (
    plate_project_sensitivity
    .groupby("candidate_pair")
    .agg(
        project_direction_preservation_fraction=(
            "direction_preserved",
            "mean",
        ),
        median_project_absolute_shift=(
            "absolute_shift",
            "median",
        ),
        maximum_project_absolute_shift=(
            "absolute_shift",
            "max",
        ),
    )
    .reset_index()
    .sort_values(
        "project_direction_preservation_fraction"
    )
)

plate_project_summary

,candidate_pair,project_direction_preservation_fraction,median_project_absolute_shift,maximum_project_absolute_shift
11,CROSS_OMIC_PAIR_13,0.703704,0.072122,0.230651
9,CROSS_OMIC_PAIR_11,0.851852,0.034341,0.131505
12,CROSS_OMIC_PAIR_14,0.888889,0.011807,0.076152
3,CROSS_OMIC_PAIR_05,0.925926,0.013455,0.048129
7,CROSS_OMIC_PAIR_09,0.962963,0.011418,0.112194
0,CROSS_OMIC_PAIR_02,0.962963,0.025271,0.298028
2,CROSS_OMIC_PAIR_04,0.962963,0.008722,0.079880
5,CROSS_OMIC_PAIR_07,0.962963,0.020194,0.047096
6,CROSS_OMIC_PAIR_08,1.000000,0.010388,0.116117
4,CROSS_OMIC_PAIR_06,1.000000,0.009100,0.053613


In [30]:
# =============================================================================
# Inspect project-level direction changes after plate adjustment
# =============================================================================

plate_direction_changes = (
    plate_project_sensitivity.loc[
        ~plate_project_sensitivity["direction_preserved"],
        [
            "candidate_pair",
            "project_id",
            "n_complete",
            "matched_unadjusted_correlation",
            "plate_adjusted_correlation",
            "absolute_shift",
        ],
    ]
    .sort_values(
        ["candidate_pair", "absolute_shift"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

plate_direction_changes

,candidate_pair,project_id,n_complete,matched_unadjusted_correlation,plate_adjusted_correlation,absolute_shift
0,CROSS_OMIC_PAIR_02,TCGA-READ,96,0.146881,-0.090116,0.236996
1,CROSS_OMIC_PAIR_04,TCGA-LUSC,369,0.003109,-0.007884,0.010993
2,CROSS_OMIC_PAIR_05,TCGA-SARC,258,-0.005600,0.006729,0.012329
3,CROSS_OMIC_PAIR_05,TCGA-TGCT,150,-0.003433,0.003173,0.006605
4,CROSS_OMIC_PAIR_07,TCGA-PCPG,179,-0.004699,0.005313,0.010011
5,CROSS_OMIC_PAIR_09,TCGA-KIRC,316,-0.007611,0.018732,0.026343
6,CROSS_OMIC_PAIR_11,TCGA-LUAD,452,-0.056807,0.057037,0.113844
7,CROSS_OMIC_PAIR_11,TCGA-LGG,513,0.045078,-0.027245,0.072323
8,CROSS_OMIC_PAIR_11,TCGA-KIRP,274,-0.026819,0.013941,0.040760
9,CROSS_OMIC_PAIR_11,TCGA-KIRC,316,0.016827,-0.013617,0.030444


In [31]:
# =============================================================================
# Decompose plate sensitivity by modality
# =============================================================================

plate_modality_records = []

for candidate in retained_candidates.itertuples(index=False):
    for project_id, project_table in candidate_plate_table.groupby(
        "project_id",
        sort=False,
    ):
        rna_residuals = residualize_categorical(
            project_table,
            candidate.rna_component,
            "rna_plate",
        )

        methylation_residuals = residualize_categorical(
            project_table,
            candidate.methylation_component,
            "methylation_plate",
        )

        valid_samples = (
            rna_residuals.notna()
            & methylation_residuals.notna()
        )

        rna_scores = project_table.loc[
            valid_samples,
            candidate.rna_component,
        ]

        methylation_scores = project_table.loc[
            valid_samples,
            candidate.methylation_component,
        ]

        plate_modality_records.append(
            {
                "candidate_pair": candidate.candidate_pair,
                "project_id": project_id,
                "unadjusted_correlation": (
                    rna_scores.corr(methylation_scores)
                ),
                "rna_plate_adjusted_correlation": (
                    rna_residuals.loc[valid_samples]
                    .corr(methylation_scores)
                ),
                "methylation_plate_adjusted_correlation": (
                    rna_scores.corr(
                        methylation_residuals.loc[valid_samples]
                    )
                ),
                "joint_plate_adjusted_correlation": (
                    rna_residuals.loc[valid_samples]
                    .corr(
                        methylation_residuals.loc[valid_samples]
                    )
                ),
            }
        )

plate_modality_correlations = pd.DataFrame(
    plate_modality_records
)

In [32]:
# =============================================================================
# Summarize plate sensitivity by modality
# =============================================================================

plate_modality_summary = (
    plate_modality_correlations
    .groupby("candidate_pair")
    .agg(
        unadjusted_median=(
            "unadjusted_correlation",
            "median",
        ),
        rna_plate_adjusted_median=(
            "rna_plate_adjusted_correlation",
            "median",
        ),
        methylation_plate_adjusted_median=(
            "methylation_plate_adjusted_correlation",
            "median",
        ),
        joint_plate_adjusted_median=(
            "joint_plate_adjusted_correlation",
            "median",
        ),
    )
    .reset_index()
)

for modality in [
    "rna_plate",
    "methylation_plate",
    "joint_plate",
]:
    plate_modality_summary[
        f"{modality}_absolute_shift"
    ] = (
        plate_modality_summary[
            f"{modality}_adjusted_median"
        ]
        - plate_modality_summary[
            "unadjusted_median"
        ]
    ).abs()

plate_modality_summary.sort_values(
    "joint_plate_absolute_shift",
    ascending=False,
)

,candidate_pair,unadjusted_median,rna_plate_adjusted_median,methylation_plate_adjusted_median,joint_plate_adjusted_median,rna_plate_absolute_shift,methylation_plate_absolute_shift,joint_plate_absolute_shift
11,CROSS_OMIC_PAIR_13,0.136554,0.024155,0.033112,0.037280,0.112398,0.103441,0.099274
0,CROSS_OMIC_PAIR_02,0.369043,0.284216,0.277758,0.293377,0.084826,0.091285,0.075666
9,CROSS_OMIC_PAIR_11,-0.103146,-0.072353,-0.075743,-0.078703,0.030793,0.027403,0.024443
4,CROSS_OMIC_PAIR_06,0.089718,0.109621,0.110232,0.112899,0.019903,0.020514,0.023180
10,CROSS_OMIC_PAIR_12,-0.101116,-0.115034,-0.117612,-0.119151,0.013918,0.016496,0.018035
3,CROSS_OMIC_PAIR_05,-0.137337,-0.146559,-0.147768,-0.154498,0.009222,0.010431,0.017161
12,CROSS_OMIC_PAIR_14,-0.098618,-0.087382,-0.072861,-0.089788,0.011236,0.025757,0.008830
5,CROSS_OMIC_PAIR_07,0.109275,0.111952,0.110371,0.117559,0.002677,0.001096,0.008284
6,CROSS_OMIC_PAIR_08,0.107848,0.099278,0.097355,0.101072,0.008570,0.010492,0.006776
1,CROSS_OMIC_PAIR_03,-0.163303,-0.171248,-0.169572,-0.169411,0.007945,0.006268,0.006108


In [33]:
# =============================================================================
# Consolidate plate-batch robustness evidence
# =============================================================================

plate_robustness_summary = (
    plate_summary[
        [
            "candidate_pair",
            "matched_unadjusted_median",
            "plate_adjusted_median",
            "absolute_shift",
        ]
    ]
    .rename(
        columns={
            "absolute_shift": "plate_median_absolute_shift",
        }
    )
    .merge(
        plate_project_summary,
        on="candidate_pair",
        how="left",
        validate="one_to_one",
    )
    .merge(
        plate_modality_summary[
            [
                "candidate_pair",
                "rna_plate_absolute_shift",
                "methylation_plate_absolute_shift",
            ]
        ],
        on="candidate_pair",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "plate_median_absolute_shift",
        ascending=False,
    )
    .reset_index(drop=True)
)

plate_robustness_summary

,candidate_pair,matched_unadjusted_median,plate_adjusted_median,plate_median_absolute_shift,project_direction_preservation_fraction,median_project_absolute_shift,maximum_project_absolute_shift,rna_plate_absolute_shift,methylation_plate_absolute_shift
0,CROSS_OMIC_PAIR_13,0.136554,0.037280,0.099274,0.703704,0.072122,0.230651,0.112398,0.103441
1,CROSS_OMIC_PAIR_02,0.369043,0.293377,0.075666,0.962963,0.025271,0.298028,0.084826,0.091285
2,CROSS_OMIC_PAIR_11,-0.103146,-0.078703,0.024443,0.851852,0.034341,0.131505,0.030793,0.027403
3,CROSS_OMIC_PAIR_06,0.089718,0.112899,0.023180,1.000000,0.009100,0.053613,0.019903,0.020514
4,CROSS_OMIC_PAIR_12,-0.101116,-0.119151,0.018035,1.000000,0.011861,0.078161,0.013918,0.016496
5,CROSS_OMIC_PAIR_05,-0.137337,-0.154498,0.017161,0.925926,0.013455,0.048129,0.009222,0.010431
6,CROSS_OMIC_PAIR_14,-0.098618,-0.089788,0.008830,0.888889,0.011807,0.076152,0.011236,0.025757
7,CROSS_OMIC_PAIR_07,0.109275,0.117559,0.008284,0.962963,0.020194,0.047096,0.002677,0.001096
8,CROSS_OMIC_PAIR_08,0.107848,0.101072,0.006776,1.000000,0.010388,0.116117,0.008570,0.010492
9,CROSS_OMIC_PAIR_03,-0.163303,-0.169411,0.006108,1.000000,0.009674,0.084541,0.007945,0.006268


In [34]:
# =============================================================================
# Define ICA stability settings
# =============================================================================

ICA_RANDOM_SEED = 42
ICA_MAX_ITER = 5_000
ICA_TOLERANCE = 1e-4
ICA_ALGORITHM = "parallel"
ICA_WHITEN = "unit-variance"
ICA_WHITEN_SOLVER = "svd"
ICA_FUNCTION = "logcosh"

RNA_ICA_COMPONENTS = (
    program_discovery_metadata["ica"]["rna_fitted_components"]
)

METHYLATION_ICA_COMPONENTS = (
    program_discovery_metadata["ica"]["methylation_fitted_components"]
)

ICA_PCA_COMPONENTS = 500

In [35]:
# =============================================================================
# Load RNA ICA source metadata
# =============================================================================

import h5py

RNA_TMM_LOGCPM_H5_PATH = (
    Paths.expression
    / "tcga_primary_tumor_rnaseq_program_discovery_tmm_logcpm.h5"
)

RNA_GENE_METADATA_PATH = (
    Paths.expression
    / "tcga_primary_tumor_rnaseq_program_discovery_filtered_gene_metadata.csv"
)

RNA_SAMPLE_METADATA_PATH = (
    Paths.expression
    / "tcga_primary_tumor_rnaseq_program_discovery_sample_metadata.csv"
)

rna_filtered_features = pd.read_csv(
    RNA_GENE_METADATA_PATH
)

rna_sample_metadata = pd.read_csv(
    RNA_SAMPLE_METADATA_PATH
)

In [36]:
# =============================================================================
# Load RNA TMM log-CPM matrix
# =============================================================================

with h5py.File(
    RNA_TMM_LOGCPM_H5_PATH,
    "r",
) as rna_tmm_logcpm_h5:
    rna_logcpm = np.empty(
        rna_tmm_logcpm_h5["logcpm"].shape,
        dtype=np.float32,
    )

    rna_tmm_logcpm_h5["logcpm"].read_direct(
        rna_logcpm
    )

rna_sample_by_gene = rna_logcpm.T

In [37]:
# =============================================================================
# Compute RNA within-project gene variances
# =============================================================================

project_labels = (
    rna_sample_metadata["project_id"]
    .to_numpy()
)

project_ids = np.sort(
    rna_sample_metadata["project_id"].unique()
)

rna_project_variances = np.empty(
    (
        len(project_ids),
        rna_logcpm.shape[0],
    ),
    dtype=np.float32,
)

for project_index, project_id in enumerate(project_ids):
    project_sample_indices = np.flatnonzero(
        project_labels == project_id
    )

    rna_project_variances[project_index] = np.var(
        rna_sample_by_gene[project_sample_indices],
        axis=0,
        ddof=1,
        dtype=np.float64,
    )

In [38]:
# =============================================================================
# Select RNA genes for ICA stability analysis
# =============================================================================

N_ICA_GENES = 5_000

rna_median_within_project_variance = np.median(
    rna_project_variances,
    axis=0,
)

rna_ica_row_indices = np.sort(
    np.argsort(
        rna_median_within_project_variance
    )[-N_ICA_GENES:]
)

rna_ica_features = (
    rna_filtered_features
    .iloc[rna_ica_row_indices]
    .copy()
    .reset_index(drop=True)
)

In [39]:
# =============================================================================
# Prepare lineage-centered RNA ICA input
# =============================================================================

rna_ica_input = np.ascontiguousarray(
    rna_sample_by_gene[:, rna_ica_row_indices],
    dtype=np.float32,
)

for project_id in project_ids:
    project_sample_indices = np.flatnonzero(
        project_labels == project_id
    )

    project_gene_means = np.mean(
        rna_ica_input[project_sample_indices],
        axis=0,
        dtype=np.float64,
    ).astype(np.float32)

    rna_ica_input[project_sample_indices] -= (
        project_gene_means
    )

In [40]:
# =============================================================================
# Fit RNA PCA representation for ICA stability
# =============================================================================

from sklearn.decomposition import PCA

rna_ica_pca = PCA(
    n_components=ICA_PCA_COMPONENTS,
    svd_solver="randomized",
    random_state=ICA_RANDOM_SEED,
)

rna_ica_pca_scores = rna_ica_pca.fit_transform(
    rna_ica_input
)

In [41]:
# =============================================================================
# Prepare RNA ICA model input
# =============================================================================

rna_ica_model_input = np.ascontiguousarray(
    rna_ica_pca_scores[:, :RNA_ICA_COMPONENTS],
    dtype=np.float64,
)

In [42]:
# =============================================================================
# Fit reference RNA ICA decomposition
# =============================================================================

from sklearn.decomposition import FastICA

rna_ica_reference = FastICA(
    n_components=RNA_ICA_COMPONENTS,
    algorithm=ICA_ALGORITHM,
    whiten=ICA_WHITEN,
    whiten_solver=ICA_WHITEN_SOLVER,
    fun=ICA_FUNCTION,
    max_iter=ICA_MAX_ITER,
    tol=ICA_TOLERANCE,
    random_state=ICA_RANDOM_SEED,
)

rna_ica_reference_scores = (
    rna_ica_reference.fit_transform(
        rna_ica_model_input
    )
)

In [43]:
# =============================================================================
# Recover oriented RNA ICA gene loadings
# =============================================================================

rna_ica_reference_gene_loadings = (
    rna_ica_reference.mixing_.T
    @ rna_ica_pca.components_[:RNA_ICA_COMPONENTS]
).astype(np.float32)

largest_loading_indices = np.argmax(
    np.abs(rna_ica_reference_gene_loadings),
    axis=1,
)

rna_ica_reference_signs = np.where(
    rna_ica_reference_gene_loadings[
        np.arange(RNA_ICA_COMPONENTS),
        largest_loading_indices,
    ] < 0,
    -1.0,
    1.0,
).astype(np.float32)

rna_ica_reference_gene_loadings *= (
    rna_ica_reference_signs[:, None]
)

rna_ica_reference_scores *= (
    rna_ica_reference_signs[None, :]
)

In [44]:
# =============================================================================
# Compare reconstructed and published RNA ICA loadings
# =============================================================================

rna_reference_loading_table = pd.DataFrame(
    rna_ica_reference_gene_loadings.T,
    columns=[
        f"RNA_IC{component_index + 1:03d}"
        for component_index in range(RNA_ICA_COMPONENTS)
    ],
)

rna_reference_loading_table["gene_id"] = (
    rna_ica_features["gene_id"].to_numpy()
)

rna_reconstruction_records = []

for component in retained_rna_components:
    comparison = (
        rna_candidate_loadings[
            ["gene_id", component]
        ]
        .merge(
            rna_reference_loading_table[
                ["gene_id", component]
            ],
            on="gene_id",
            how="inner",
            suffixes=(
                "_published",
                "_reconstructed",
            ),
        )
    )

    rna_reconstruction_records.append(
        {
            "component": component,
            "loading_correlation": comparison[
                f"{component}_published"
            ].corr(
                comparison[
                    f"{component}_reconstructed"
                ]
            ),
        }
    )

rna_reconstruction_summary = (
    pd.DataFrame(rna_reconstruction_records)
    .sort_values("loading_correlation")
    .reset_index(drop=True)
)

rna_reconstruction_summary

,component,loading_correlation
0,RNA_IC150,1.0
1,RNA_IC083,1.0
2,RNA_IC169,1.0
3,RNA_IC193,1.0
4,RNA_IC175,1.0
5,RNA_IC050,1.0
6,RNA_IC001,1.0
7,RNA_IC184,1.0
8,RNA_IC151,1.0
9,RNA_IC158,1.0


In [45]:
# =============================================================================
# Define RNA ICA seed-stability refits
# =============================================================================

RNA_ICA_STABILITY_SEEDS = (
    7,
    19,
    31,
    53,
    71,
    89,
    107,
    131,
    149,
    173,
)

In [46]:
# =============================================================================
# Refit RNA ICA across alternative random seeds
# =============================================================================

rna_ica_seed_loadings = {}
rna_ica_seed_iterations = {}

for seed in RNA_ICA_STABILITY_SEEDS:
    ica_refit = FastICA(
        n_components=RNA_ICA_COMPONENTS,
        algorithm=ICA_ALGORITHM,
        whiten=ICA_WHITEN,
        whiten_solver=ICA_WHITEN_SOLVER,
        fun=ICA_FUNCTION,
        max_iter=ICA_MAX_ITER,
        tol=ICA_TOLERANCE,
        random_state=seed,
    )

    ica_refit.fit(
        rna_ica_model_input
    )

    rna_ica_seed_loadings[seed] = (
        ica_refit.mixing_.T
        @ rna_ica_pca.components_[:RNA_ICA_COMPONENTS]
    ).astype(np.float32)

    rna_ica_seed_iterations[seed] = (
        ica_refit.n_iter_
    )

In [47]:
# =============================================================================
# Match RNA ICA seed refits to the reference decomposition
# =============================================================================

from scipy.optimize import linear_sum_assignment

reference_centered = (
    rna_ica_reference_gene_loadings
    - rna_ica_reference_gene_loadings.mean(
        axis=1,
        keepdims=True,
    )
)

reference_normalized = (
    reference_centered
    / np.linalg.norm(
        reference_centered,
        axis=1,
        keepdims=True,
    )
)

rna_ica_seed_match_records = []

for seed, refit_loadings in rna_ica_seed_loadings.items():
    refit_centered = (
        refit_loadings
        - refit_loadings.mean(
            axis=1,
            keepdims=True,
        )
    )

    refit_normalized = (
        refit_centered
        / np.linalg.norm(
            refit_centered,
            axis=1,
            keepdims=True,
        )
    )

    correlation_matrix = (
        reference_normalized
        @ refit_normalized.T
    )

    reference_indices, refit_indices = (
        linear_sum_assignment(
            -np.abs(correlation_matrix)
        )
    )

    matched_refit_indices = dict(
        zip(reference_indices, refit_indices)
    )

    for component in retained_rna_components:
        reference_index = int(
            component.removeprefix("RNA_IC")
        ) - 1

        refit_index = matched_refit_indices[
            reference_index
        ]

        correlation = correlation_matrix[
            reference_index,
            refit_index,
        ]

        rna_ica_seed_match_records.append(
            {
                "seed": seed,
                "component": component,
                "matched_refit_component": (
                    f"RNA_IC{refit_index + 1:03d}"
                ),
                "loading_correlation": correlation,
                "absolute_loading_correlation": abs(
                    correlation
                ),
                "ica_iterations": (
                    rna_ica_seed_iterations[seed]
                ),
            }
        )

rna_ica_seed_match_table = pd.DataFrame(
    rna_ica_seed_match_records
)

In [48]:
# =============================================================================
# Summarize RNA ICA seed stability
# =============================================================================

rna_ica_seed_stability_summary = (
    rna_ica_seed_match_table
    .groupby("component", as_index=False)
    .agg(
        median_absolute_loading_correlation=(
            "absolute_loading_correlation",
            "median",
        ),
        minimum_absolute_loading_correlation=(
            "absolute_loading_correlation",
            "min",
        ),
        maximum_absolute_loading_correlation=(
            "absolute_loading_correlation",
            "max",
        ),
    )
    .sort_values(
        "minimum_absolute_loading_correlation"
    )
    .reset_index(drop=True)
)

rna_ica_seed_stability_summary

,component,median_absolute_loading_correlation,minimum_absolute_loading_correlation,maximum_absolute_loading_correlation
0,RNA_IC169,0.393281,0.098197,0.633002
1,RNA_IC001,0.448719,0.151901,0.635125
2,RNA_IC050,0.494758,0.226285,0.683501
3,RNA_IC083,0.733854,0.337683,0.951072
4,RNA_IC193,0.448745,0.340445,0.625756
5,RNA_IC151,0.623267,0.399135,0.793412
6,RNA_IC175,0.650641,0.566954,0.734221
7,RNA_IC158,0.983791,0.804359,0.990442
8,RNA_IC184,0.940041,0.871664,0.966505
9,RNA_IC150,0.989194,0.982535,0.991990


In [49]:
# =============================================================================
# Summarize RNA ICA refit convergence
# =============================================================================

rna_ica_convergence_summary = pd.DataFrame(
    {
        "seed": list(rna_ica_seed_iterations),
        "ica_iterations": list(
            rna_ica_seed_iterations.values()
        ),
    }
)

rna_ica_convergence_summary["converged"] = (
    rna_ica_convergence_summary["ica_iterations"]
    < ICA_MAX_ITER
)

rna_ica_convergence_summary

,seed,ica_iterations,converged
0,7,161,True
1,19,146,True
2,31,87,True
3,53,102,True
4,71,109,True
5,89,143,True
6,107,97,True
7,131,94,True
8,149,116,True
9,173,102,True


In [50]:
# =============================================================================
# Evaluate unrestricted RNA ICA component recovery
# =============================================================================

rna_ica_unrestricted_match_records = []

for seed, refit_loadings in rna_ica_seed_loadings.items():
    refit_centered = (
        refit_loadings
        - refit_loadings.mean(
            axis=1,
            keepdims=True,
        )
    )

    refit_normalized = (
        refit_centered
        / np.linalg.norm(
            refit_centered,
            axis=1,
            keepdims=True,
        )
    )

    correlation_matrix = (
        reference_normalized
        @ refit_normalized.T
    )

    for component in retained_rna_components:
        reference_index = (
            int(component.removeprefix("RNA_IC")) - 1
        )

        absolute_correlations = np.abs(
            correlation_matrix[reference_index]
        )

        best_refit_index = np.argmax(
            absolute_correlations
        )

        rna_ica_unrestricted_match_records.append(
            {
                "seed": seed,
                "component": component,
                "best_absolute_loading_correlation": (
                    absolute_correlations[
                        best_refit_index
                    ]
                ),
            }
        )

rna_ica_unrestricted_match_table = pd.DataFrame(
    rna_ica_unrestricted_match_records
)

In [51]:
# =============================================================================
# Compare constrained and unrestricted RNA ICA matching
# =============================================================================

rna_ica_unrestricted_summary = (
    rna_ica_unrestricted_match_table
    .groupby("component", as_index=False)
    .agg(
        unrestricted_median_absolute_correlation=(
            "best_absolute_loading_correlation",
            "median",
        ),
        unrestricted_minimum_absolute_correlation=(
            "best_absolute_loading_correlation",
            "min",
        ),
    )
)

rna_ica_matching_comparison = (
    rna_ica_seed_stability_summary
    .merge(
        rna_ica_unrestricted_summary,
        on="component",
        how="left",
        validate="one_to_one",
    )
)

rna_ica_matching_comparison[
    "median_matching_gain"
] = (
    rna_ica_matching_comparison[
        "unrestricted_median_absolute_correlation"
    ]
    - rna_ica_matching_comparison[
        "median_absolute_loading_correlation"
    ]
)

rna_ica_matching_comparison[
    [
        "component",
        "median_absolute_loading_correlation",
        "unrestricted_median_absolute_correlation",
        "median_matching_gain",
        "minimum_absolute_loading_correlation",
        "unrestricted_minimum_absolute_correlation",
    ]
].sort_values(
    "unrestricted_median_absolute_correlation"
)

,component,median_absolute_loading_correlation,unrestricted_median_absolute_correlation,median_matching_gain,minimum_absolute_loading_correlation,unrestricted_minimum_absolute_correlation
0,RNA_IC169,0.393281,0.393281,0.000000,0.098197,0.336462
4,RNA_IC193,0.448745,0.472625,0.023880,0.340445,0.380149
2,RNA_IC050,0.494758,0.497790,0.003032,0.226285,0.371602
1,RNA_IC001,0.448719,0.497957,0.049238,0.151901,0.408670
5,RNA_IC151,0.623267,0.623267,0.000000,0.399135,0.399135
6,RNA_IC175,0.650641,0.650641,0.000000,0.566954,0.566954
3,RNA_IC083,0.733854,0.733854,0.000000,0.337683,0.447309
8,RNA_IC184,0.940041,0.940041,0.000000,0.871664,0.871664
7,RNA_IC158,0.983791,0.983791,0.000000,0.804359,0.848907
9,RNA_IC150,0.989194,0.989194,0.000000,0.982535,0.982535


In [52]:
# =============================================================================
# Define RNA ICA stratified-subsampling settings
# =============================================================================

RNA_ICA_SUBSAMPLE_FRACTION = 0.80

RNA_ICA_SUBSAMPLE_SEEDS = (
    11,
    23,
    37,
    47,
    61,
    73,
    97,
    109,
    127,
    157,
)

rna_ica_subsample_indices = {}

for seed in RNA_ICA_SUBSAMPLE_SEEDS:
    rng = np.random.default_rng(seed)
    selected_indices = []

    for project_id in project_ids:
        project_indices = np.flatnonzero(
            project_labels == project_id
        )

        n_selected = max(
            2,
            int(
                np.floor(
                    len(project_indices)
                    * RNA_ICA_SUBSAMPLE_FRACTION
                )
            ),
        )

        selected_indices.extend(
            rng.choice(
                project_indices,
                size=n_selected,
                replace=False,
            )
        )

    rna_ica_subsample_indices[seed] = np.sort(
        np.asarray(
            selected_indices,
            dtype=np.int64,
        )
    )

In [53]:
# =============================================================================
# Refit RNA ICA across stratified sample subsamples
# =============================================================================

rna_ica_subsample_loadings = {}
rna_ica_subsample_iterations = {}

for seed, sample_indices in rna_ica_subsample_indices.items():
    ica_refit = FastICA(
        n_components=RNA_ICA_COMPONENTS,
        algorithm=ICA_ALGORITHM,
        whiten=ICA_WHITEN,
        whiten_solver=ICA_WHITEN_SOLVER,
        fun=ICA_FUNCTION,
        max_iter=ICA_MAX_ITER,
        tol=ICA_TOLERANCE,
        random_state=ICA_RANDOM_SEED,
    )

    ica_refit.fit(
        rna_ica_model_input[sample_indices]
    )

    rna_ica_subsample_loadings[seed] = (
        ica_refit.mixing_.T
        @ rna_ica_pca.components_[:RNA_ICA_COMPONENTS]
    ).astype(np.float32)

    rna_ica_subsample_iterations[seed] = (
        ica_refit.n_iter_
    )

In [54]:
# =============================================================================
# Match RNA ICA subsample refits to the reference decomposition
# =============================================================================

rna_ica_subsample_match_records = []

for seed, refit_loadings in rna_ica_subsample_loadings.items():
    refit_centered = (
        refit_loadings
        - refit_loadings.mean(
            axis=1,
            keepdims=True,
        )
    )

    refit_normalized = (
        refit_centered
        / np.linalg.norm(
            refit_centered,
            axis=1,
            keepdims=True,
        )
    )

    correlation_matrix = (
        reference_normalized
        @ refit_normalized.T
    )

    reference_indices, refit_indices = (
        linear_sum_assignment(
            -np.abs(correlation_matrix)
        )
    )

    matched_refit_indices = dict(
        zip(reference_indices, refit_indices)
    )

    for component in retained_rna_components:
        reference_index = (
            int(component.removeprefix("RNA_IC")) - 1
        )

        refit_index = matched_refit_indices[
            reference_index
        ]

        correlation = correlation_matrix[
            reference_index,
            refit_index,
        ]

        rna_ica_subsample_match_records.append(
            {
                "seed": seed,
                "component": component,
                "matched_refit_component": (
                    f"RNA_IC{refit_index + 1:03d}"
                ),
                "loading_correlation": correlation,
                "absolute_loading_correlation": abs(
                    correlation
                ),
                "ica_iterations": (
                    rna_ica_subsample_iterations[seed]
                ),
            }
        )

rna_ica_subsample_match_table = pd.DataFrame(
    rna_ica_subsample_match_records
)

In [55]:
# =============================================================================
# Summarize RNA ICA subsample stability
# =============================================================================

rna_ica_subsample_stability_summary = (
    rna_ica_subsample_match_table
    .groupby("component", as_index=False)
    .agg(
        median_absolute_loading_correlation=(
            "absolute_loading_correlation",
            "median",
        ),
        minimum_absolute_loading_correlation=(
            "absolute_loading_correlation",
            "min",
        ),
        maximum_absolute_loading_correlation=(
            "absolute_loading_correlation",
            "max",
        ),
        maximum_ica_iterations=(
            "ica_iterations",
            "max",
        ),
    )
    .sort_values(
        "minimum_absolute_loading_correlation"
    )
    .reset_index(drop=True)
)

rna_ica_subsample_stability_summary

,component,median_absolute_loading_correlation,minimum_absolute_loading_correlation,maximum_absolute_loading_correlation,maximum_ica_iterations
0,RNA_IC169,0.336027,0.182642,0.464137,169
1,RNA_IC193,0.381122,0.244317,0.520419,169
2,RNA_IC050,0.385271,0.245557,0.729797,169
3,RNA_IC151,0.583573,0.367148,0.820244,169
4,RNA_IC083,0.817579,0.369531,0.929917,169
5,RNA_IC001,0.455537,0.370959,0.569877,169
6,RNA_IC175,0.571628,0.453695,0.692697,169
7,RNA_IC150,0.980672,0.629612,0.987261,169
8,RNA_IC184,0.950575,0.677766,0.967156,169
9,RNA_IC158,0.984331,0.805536,0.990556,169


In [56]:
# =============================================================================
# Integrate RNA ICA decomposition-stability evidence
# =============================================================================

rna_ica_stability_summary = (
    rna_ica_seed_stability_summary[
        [
            "component",
            "median_absolute_loading_correlation",
            "minimum_absolute_loading_correlation",
        ]
    ]
    .rename(
        columns={
            "median_absolute_loading_correlation": (
                "seed_median_absolute_correlation"
            ),
            "minimum_absolute_loading_correlation": (
                "seed_minimum_absolute_correlation"
            ),
        }
    )
    .merge(
        rna_ica_subsample_stability_summary[
            [
                "component",
                "median_absolute_loading_correlation",
                "minimum_absolute_loading_correlation",
            ]
        ].rename(
            columns={
                "median_absolute_loading_correlation": (
                    "subsample_median_absolute_correlation"
                ),
                "minimum_absolute_loading_correlation": (
                    "subsample_minimum_absolute_correlation"
                ),
            }
        ),
        on="component",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "subsample_median_absolute_correlation"
    )
    .reset_index(drop=True)
)

rna_ica_stability_summary

,component,seed_median_absolute_correlation,seed_minimum_absolute_correlation,subsample_median_absolute_correlation,subsample_minimum_absolute_correlation
0,RNA_IC169,0.393281,0.098197,0.336027,0.182642
1,RNA_IC193,0.448745,0.340445,0.381122,0.244317
2,RNA_IC050,0.494758,0.226285,0.385271,0.245557
3,RNA_IC001,0.448719,0.151901,0.455537,0.370959
4,RNA_IC175,0.650641,0.566954,0.571628,0.453695
5,RNA_IC151,0.623267,0.399135,0.583573,0.367148
6,RNA_IC083,0.733854,0.337683,0.817579,0.369531
7,RNA_IC184,0.940041,0.871664,0.950575,0.677766
8,RNA_IC150,0.989194,0.982535,0.980672,0.629612
9,RNA_IC158,0.983791,0.804359,0.984331,0.805536


In [57]:
# =============================================================================
# Load methylation ICA source representation
# =============================================================================

METHYLATION_M_VALUES_PATH = (
    Paths.methylation
    / "tcga_primary_tumor_shared_methylation_program_discovery_m_values.npy"
)

methylation_m_values = np.load(
    METHYLATION_M_VALUES_PATH,
    mmap_mode="r",
)

methylation_variable_probe_indices = (
    methylation_candidate_loadings[
        "model_matrix_row_index"
    ].to_numpy(dtype=np.int64)
)

methylation_hm450_sample_indices = (
    methylation_candidate_scores[
        "final_sample_column_index"
    ].to_numpy(dtype=np.int64)
)

In [58]:
# =============================================================================
# Reconstruct methylation ICA feature matrix
# =============================================================================

methylation_hm450_variable_matrix = np.asarray(
    methylation_m_values[
        methylation_variable_probe_indices
    ][:, methylation_hm450_sample_indices].T,
    dtype=np.float32,
)

methylation_hm450_variable_matrix = np.ascontiguousarray(
    methylation_hm450_variable_matrix
)

In [59]:
# =============================================================================
# Center methylation ICA input within project
# =============================================================================

methylation_hm450_project_labels = (
    methylation_candidate_scores[
        "project_id"
    ].to_numpy()
)

methylation_hm450_project_ids = np.sort(
    np.unique(methylation_hm450_project_labels)
)

methylation_hm450_ica_input = np.array(
    methylation_hm450_variable_matrix,
    dtype=np.float32,
    order="C",
    copy=True,
)

for project_id in methylation_hm450_project_ids:
    project_sample_indices = np.flatnonzero(
        methylation_hm450_project_labels == project_id
    )

    project_probe_means = np.mean(
        methylation_hm450_ica_input[
            project_sample_indices
        ],
        axis=0,
        dtype=np.float64,
    ).astype(np.float32)

    methylation_hm450_ica_input[
        project_sample_indices
    ] -= project_probe_means

In [60]:
# =============================================================================
# Fit methylation pre-ICA PCA
# =============================================================================

methylation_hm450_ica_pca = PCA(
    n_components=ICA_PCA_COMPONENTS,
    svd_solver="randomized",
    random_state=ICA_RANDOM_SEED,
)

methylation_hm450_ica_pca_scores = (
    methylation_hm450_ica_pca.fit_transform(
        methylation_hm450_ica_input
    )
)

In [61]:
# =============================================================================
# Build methylation ICA model input
# =============================================================================

methylation_hm450_ica_model_input = np.ascontiguousarray(
    methylation_hm450_ica_pca_scores[
        :,
        :METHYLATION_ICA_COMPONENTS,
    ],
    dtype=np.float64,
)

In [62]:
# =============================================================================
# Fit reference methylation ICA
# =============================================================================

methylation_ica_reference = FastICA(
    n_components=METHYLATION_ICA_COMPONENTS,
    algorithm=ICA_ALGORITHM,
    whiten=ICA_WHITEN,
    whiten_solver=ICA_WHITEN_SOLVER,
    fun=ICA_FUNCTION,
    max_iter=ICA_MAX_ITER,
    tol=ICA_TOLERANCE,
    random_state=ICA_RANDOM_SEED,
)

methylation_ica_reference_scores = (
    methylation_ica_reference.fit_transform(
        methylation_hm450_ica_model_input
    )
)

In [63]:
# =============================================================================
# Recover and orient reference methylation ICA probe loadings
# =============================================================================

methylation_ica_reference_probe_loadings = (
    methylation_ica_reference.mixing_.T
    @ methylation_hm450_ica_pca.components_[
        :METHYLATION_ICA_COMPONENTS
    ]
).astype(np.float32)

largest_loading_indices = np.argmax(
    np.abs(methylation_ica_reference_probe_loadings),
    axis=1,
)

methylation_ica_reference_signs = np.where(
    methylation_ica_reference_probe_loadings[
        np.arange(METHYLATION_ICA_COMPONENTS),
        largest_loading_indices,
    ] < 0,
    -1.0,
    1.0,
).astype(np.float32)

methylation_ica_reference_probe_loadings *= (
    methylation_ica_reference_signs[:, None]
)

methylation_ica_reference_scores *= (
    methylation_ica_reference_signs[None, :]
)

In [64]:
# =============================================================================
# Compare reconstructed methylation ICA loadings with published loadings
# =============================================================================

methylation_reference_comparison_records = []

for component in retained_methylation_components:
    component_index = (
        int(component.removeprefix("METH_IC")) - 1
    )

    loading_correlation = np.corrcoef(
        methylation_ica_reference_probe_loadings[
            component_index
        ],
        methylation_candidate_loadings[
            component
        ].to_numpy(dtype=np.float32),
    )[0, 1]

    methylation_reference_comparison_records.append(
        {
            "component": component,
            "loading_correlation": loading_correlation,
        }
    )

methylation_reference_comparison = pd.DataFrame(
    methylation_reference_comparison_records
)

methylation_reference_comparison

,component,loading_correlation
0,METH_IC232,1.0
1,METH_IC169,1.0
2,METH_IC128,1.0
3,METH_IC023,1.0
4,METH_IC013,1.0
5,METH_IC234,1.0
6,METH_IC050,1.0
7,METH_IC107,1.0
8,METH_IC241,1.0
9,METH_IC109,1.0


In [65]:
# =============================================================================
# Define methylation ICA seed-stability settings
# =============================================================================

METHYLATION_ICA_STABILITY_SEEDS = (
    7,
    19,
    31,
    53,
    71,
    89,
    107,
    131,
    149,
    173,
)

retained_methylation_component_indices = np.array(
    [
        int(component.removeprefix("METH_IC")) - 1
        for component in retained_methylation_components
    ],
    dtype=np.int64,
)

In [66]:
# =============================================================================
# Refit methylation ICA across random seeds
# =============================================================================

methylation_seed_stability_records = []

reference_loadings = (
    methylation_ica_reference_probe_loadings.astype(
        np.float64,
        copy=False,
    )
)

reference_loadings_centered = (
    reference_loadings
    - reference_loadings.mean(axis=1, keepdims=True)
)

reference_loading_norms = np.linalg.norm(
    reference_loadings_centered,
    axis=1,
    keepdims=True,
)

reference_loadings_normalized = (
    reference_loadings_centered
    / reference_loading_norms
)

for seed in METHYLATION_ICA_STABILITY_SEEDS:
    seed_ica = FastICA(
        n_components=METHYLATION_ICA_COMPONENTS,
        algorithm=ICA_ALGORITHM,
        whiten=ICA_WHITEN,
        whiten_solver=ICA_WHITEN_SOLVER,
        fun=ICA_FUNCTION,
        max_iter=ICA_MAX_ITER,
        tol=ICA_TOLERANCE,
        random_state=seed,
    )

    seed_ica.fit(
        methylation_hm450_ica_model_input
    )

    seed_probe_loadings = (
        seed_ica.mixing_.T
        @ methylation_hm450_ica_pca.components_[
            :METHYLATION_ICA_COMPONENTS
        ]
    )

    seed_probe_loadings -= (
        seed_probe_loadings.mean(
            axis=1,
            keepdims=True,
        )
    )

    seed_probe_loadings /= np.linalg.norm(
        seed_probe_loadings,
        axis=1,
        keepdims=True,
    )

    loading_correlations = (
        reference_loadings_normalized
        @ seed_probe_loadings.T
    )

    reference_indices, seed_indices = (
        linear_sum_assignment(
            -np.abs(loading_correlations)
        )
    )

    matched_seed_indices = np.empty(
        METHYLATION_ICA_COMPONENTS,
        dtype=np.int64,
    )

    matched_seed_indices[
        reference_indices
    ] = seed_indices

    for component, reference_index in zip(
        retained_methylation_components,
        retained_methylation_component_indices,
    ):
        seed_index = matched_seed_indices[
            reference_index
        ]

        methylation_seed_stability_records.append(
            {
                "seed": seed,
                "component": component,
                "matched_component": (
                    f"METH_IC{seed_index + 1:03d}"
                ),
                "absolute_loading_correlation": abs(
                    loading_correlations[
                        reference_index,
                        seed_index,
                    ]
                ),
                "n_iter": seed_ica.n_iter_,
            }
        )

In [67]:
# =============================================================================
# Summarize methylation ICA seed stability
# =============================================================================

methylation_seed_stability = pd.DataFrame(
    methylation_seed_stability_records
)

methylation_seed_stability_summary = (
    methylation_seed_stability
    .groupby(
        "component",
        observed=True,
    )["absolute_loading_correlation"]
    .agg(
        seed_median="median",
        seed_min="min",
        seed_max="max",
    )
    .reset_index()
    .sort_values(
        "seed_median",
        ascending=True,
    )
    .reset_index(drop=True)
)

methylation_seed_stability_summary

,component,seed_median,seed_min,seed_max
0,METH_IC234,0.295562,0.205798,0.504638
1,METH_IC050,0.312480,0.198219,0.451757
2,METH_IC107,0.337980,0.209159,0.486561
3,METH_IC023,0.368214,0.327553,0.462366
4,METH_IC241,0.369872,0.242210,0.528949
5,METH_IC013,0.380165,0.272103,0.476381
6,METH_IC169,0.462809,0.362390,0.654262
7,METH_IC033,0.676084,0.329697,0.947769
8,METH_IC109,0.860798,0.474996,0.974192
9,METH_IC232,0.919843,0.830950,0.959729


In [68]:
# =============================================================================
# Define methylation ICA sample-subsampling settings
# =============================================================================

METHYLATION_ICA_SUBSAMPLE_FRACTION = 0.80

METHYLATION_ICA_SUBSAMPLE_SEEDS = (
    11,
    23,
    37,
    47,
    61,
    73,
    97,
    109,
    127,
    157,
)

In [69]:
# =============================================================================
# Refit methylation ICA across stratified sample subsamples
# =============================================================================

methylation_subsample_stability_records = []

for seed in METHYLATION_ICA_SUBSAMPLE_SEEDS:
    rng = np.random.default_rng(seed)

    subsample_indices = []

    for project_id in methylation_hm450_project_ids:
        project_indices = np.flatnonzero(
            methylation_hm450_project_labels == project_id
        )

        subsample_size = max(
            2,
            int(
                np.floor(
                    len(project_indices)
                    * METHYLATION_ICA_SUBSAMPLE_FRACTION
                )
            ),
        )

        subsample_indices.extend(
            rng.choice(
                project_indices,
                size=subsample_size,
                replace=False,
            )
        )

    subsample_indices = np.sort(
        np.asarray(
            subsample_indices,
            dtype=np.int64,
        )
    )

    subsample_ica = FastICA(
        n_components=METHYLATION_ICA_COMPONENTS,
        algorithm=ICA_ALGORITHM,
        whiten=ICA_WHITEN,
        whiten_solver=ICA_WHITEN_SOLVER,
        fun=ICA_FUNCTION,
        max_iter=ICA_MAX_ITER,
        tol=ICA_TOLERANCE,
        random_state=ICA_RANDOM_SEED,
    )

    subsample_ica.fit(
        methylation_hm450_ica_model_input[
            subsample_indices
        ]
    )

    subsample_probe_loadings = (
        subsample_ica.mixing_.T
        @ methylation_hm450_ica_pca.components_[
            :METHYLATION_ICA_COMPONENTS
        ]
    )

    subsample_probe_loadings -= (
        subsample_probe_loadings.mean(
            axis=1,
            keepdims=True,
        )
    )

    subsample_probe_loadings /= np.linalg.norm(
        subsample_probe_loadings,
        axis=1,
        keepdims=True,
    )

    loading_correlations = (
        reference_loadings_normalized
        @ subsample_probe_loadings.T
    )

    reference_indices, subsample_component_indices = (
        linear_sum_assignment(
            -np.abs(loading_correlations)
        )
    )

    matched_subsample_indices = np.empty(
        METHYLATION_ICA_COMPONENTS,
        dtype=np.int64,
    )

    matched_subsample_indices[
        reference_indices
    ] = subsample_component_indices

    for component, reference_index in zip(
        retained_methylation_components,
        retained_methylation_component_indices,
    ):
        subsample_component_index = (
            matched_subsample_indices[
                reference_index
            ]
        )

        methylation_subsample_stability_records.append(
            {
                "seed": seed,
                "component": component,
                "absolute_loading_correlation": abs(
                    loading_correlations[
                        reference_index,
                        subsample_component_index,
                    ]
                ),
                "n_iter": subsample_ica.n_iter_,
            }
        )

In [70]:
# =============================================================================
# Summarize methylation ICA sample-subsampling stability
# =============================================================================

methylation_subsample_stability = pd.DataFrame(
    methylation_subsample_stability_records
)

methylation_subsample_stability_summary = (
    methylation_subsample_stability
    .groupby(
        "component",
        observed=True,
    )["absolute_loading_correlation"]
    .agg(
        subsample_median="median",
        subsample_min="min",
        subsample_max="max",
    )
    .reset_index()
    .sort_values(
        "subsample_median",
        ascending=True,
    )
    .reset_index(drop=True)
)

methylation_subsample_stability_summary

,component,subsample_median,subsample_min,subsample_max
0,METH_IC107,0.323272,0.237490,0.429750
1,METH_IC234,0.328067,0.168690,0.468872
2,METH_IC050,0.337208,0.218578,0.486528
3,METH_IC241,0.356044,0.280652,0.463048
4,METH_IC013,0.406513,0.313721,0.525111
5,METH_IC023,0.411761,0.301415,0.546298
6,METH_IC169,0.538696,0.388898,0.863417
7,METH_IC033,0.625671,0.271143,0.911588
8,METH_IC109,0.851007,0.318011,0.967748
9,METH_IC232,0.871937,0.307958,0.927385


In [71]:
# =============================================================================
# Integrate methylation ICA stability summaries
# =============================================================================

methylation_ica_stability_summary = (
    methylation_seed_stability_summary
    .merge(
        methylation_subsample_stability_summary,
        on="component",
        how="inner",
    )
    .sort_values(
        [
            "seed_median",
            "subsample_median",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

methylation_ica_stability_summary

,component,seed_median,seed_min,seed_max,subsample_median,subsample_min,subsample_max
0,METH_IC128,0.972962,0.351631,0.994019,0.948443,0.311758,0.986546
1,METH_IC232,0.919843,0.830950,0.959729,0.871937,0.307958,0.927385
2,METH_IC109,0.860798,0.474996,0.974192,0.851007,0.318011,0.967748
3,METH_IC033,0.676084,0.329697,0.947769,0.625671,0.271143,0.911588
4,METH_IC169,0.462809,0.362390,0.654262,0.538696,0.388898,0.863417
5,METH_IC013,0.380165,0.272103,0.476381,0.406513,0.313721,0.525111
6,METH_IC241,0.369872,0.242210,0.528949,0.356044,0.280652,0.463048
7,METH_IC023,0.368214,0.327553,0.462366,0.411761,0.301415,0.546298
8,METH_IC107,0.337980,0.209159,0.486561,0.323272,0.237490,0.429750
9,METH_IC050,0.312480,0.198219,0.451757,0.337208,0.218578,0.486528


In [72]:
# =============================================================================
# Integrate RNA and methylation ICA stability by candidate pair
# =============================================================================

candidate_pair_decomposition_stability = (
    retained_candidates[
        [
            "candidate_pair",
            "rna_component",
            "methylation_component",
        ]
    ]
    .merge(
        rna_ica_stability_summary.rename(
            columns={
                "component": "rna_component",
                "seed_median_absolute_correlation": "rna_seed_median",
                "seed_minimum_absolute_correlation": "rna_seed_min",
                "subsample_median_absolute_correlation": "rna_subsample_median",
                "subsample_minimum_absolute_correlation": "rna_subsample_min",
            }
        ),
        on="rna_component",
        how="left",
    )
    .merge(
        methylation_ica_stability_summary.rename(
            columns={
                "component": "methylation_component",
                "seed_median": "methylation_seed_median",
                "seed_min": "methylation_seed_min",
                "subsample_median": "methylation_subsample_median",
                "subsample_min": "methylation_subsample_min",
            }
        )[
            [
                "methylation_component",
                "methylation_seed_median",
                "methylation_seed_min",
                "methylation_subsample_median",
                "methylation_subsample_min",
            ]
        ],
        on="methylation_component",
        how="left",
    )
)

candidate_pair_decomposition_stability

,candidate_pair,rna_component,methylation_component,rna_seed_median,rna_seed_min,rna_subsample_median,rna_subsample_min,methylation_seed_median,methylation_seed_min,methylation_subsample_median,methylation_subsample_min
0,CROSS_OMIC_PAIR_02,RNA_IC083,METH_IC232,0.733854,0.337683,0.817579,0.369531,0.919843,0.830950,0.871937,0.307958
1,CROSS_OMIC_PAIR_03,RNA_IC184,METH_IC169,0.940041,0.871664,0.950575,0.677766,0.462809,0.362390,0.538696,0.388898
2,CROSS_OMIC_PAIR_04,RNA_IC150,METH_IC128,0.989194,0.982535,0.980672,0.629612,0.972962,0.351631,0.948443,0.311758
3,CROSS_OMIC_PAIR_05,RNA_IC169,METH_IC023,0.393281,0.098197,0.336027,0.182642,0.368214,0.327553,0.411761,0.301415
4,CROSS_OMIC_PAIR_06,RNA_IC175,METH_IC013,0.650641,0.566954,0.571628,0.453695,0.380165,0.272103,0.406513,0.313721
5,CROSS_OMIC_PAIR_07,RNA_IC050,METH_IC234,0.494758,0.226285,0.385271,0.245557,0.295562,0.205798,0.328067,0.168690
6,CROSS_OMIC_PAIR_08,RNA_IC151,METH_IC050,0.623267,0.399135,0.583573,0.367148,0.312480,0.198219,0.337208,0.218578
7,CROSS_OMIC_PAIR_09,RNA_IC001,METH_IC107,0.448719,0.151901,0.455537,0.370959,0.337980,0.209159,0.323272,0.237490
8,CROSS_OMIC_PAIR_10,RNA_IC001,METH_IC241,0.448719,0.151901,0.455537,0.370959,0.369872,0.242210,0.356044,0.280652
9,CROSS_OMIC_PAIR_11,RNA_IC193,METH_IC109,0.448745,0.340445,0.381122,0.244317,0.860798,0.474996,0.851007,0.318011


In [73]:
# =============================================================================
# Summarize candidate-pair decomposition stability
# =============================================================================

candidate_pair_decomposition_summary = (
    candidate_pair_decomposition_stability[
        [
            "candidate_pair",
            "rna_component",
            "methylation_component",
            "rna_seed_median",
            "rna_subsample_median",
            "methylation_seed_median",
            "methylation_subsample_median",
        ]
    ]
    .merge(
        retained_candidates[
            [
                "candidate_pair",
                "median_project_correlation",
                "direction_consistency",
                "minimum_nmf_concordance",
            ]
        ],
        on="candidate_pair",
        how="left",
    )
)

candidate_pair_decomposition_summary

,candidate_pair,rna_component,methylation_component,rna_seed_median,rna_subsample_median,methylation_seed_median,methylation_subsample_median,median_project_correlation,direction_consistency,minimum_nmf_concordance
0,CROSS_OMIC_PAIR_02,RNA_IC083,METH_IC232,0.733854,0.817579,0.919843,0.871937,0.321344,0.90625,0.072895
1,CROSS_OMIC_PAIR_03,RNA_IC184,METH_IC169,0.940041,0.950575,0.462809,0.538696,-0.179417,0.87500,0.103034
2,CROSS_OMIC_PAIR_04,RNA_IC150,METH_IC128,0.989194,0.980672,0.972962,0.948443,0.133099,0.84375,0.316420
3,CROSS_OMIC_PAIR_05,RNA_IC169,METH_IC023,0.393281,0.336027,0.368214,0.411761,-0.130233,0.84375,0.068668
4,CROSS_OMIC_PAIR_06,RNA_IC175,METH_IC013,0.650641,0.571628,0.380165,0.406513,0.112888,0.78125,0.079366
5,CROSS_OMIC_PAIR_07,RNA_IC050,METH_IC234,0.494758,0.385271,0.295562,0.328067,0.110080,0.78125,0.073354
6,CROSS_OMIC_PAIR_08,RNA_IC151,METH_IC050,0.623267,0.583573,0.312480,0.337208,0.109256,0.87500,0.089150
7,CROSS_OMIC_PAIR_09,RNA_IC001,METH_IC107,0.448719,0.455537,0.337980,0.323272,0.107940,0.75000,0.066818
8,CROSS_OMIC_PAIR_10,RNA_IC001,METH_IC241,0.448719,0.455537,0.369872,0.356044,0.107284,0.90625,0.048450
9,CROSS_OMIC_PAIR_11,RNA_IC193,METH_IC109,0.448745,0.381122,0.860798,0.851007,-0.105210,0.81250,0.175624


In [74]:
# =============================================================================
# Integrate candidate-program robustness evidence
# =============================================================================

candidate_program_robustness_summary = (
    candidate_pair_decomposition_summary
    .merge(
        lopo_summary[
            [
                "candidate_pair",
                "most_influential_project",
                "lopo_median_correlation",
                "absolute_shift",
                "direction_preserved_all",
            ]
        ].rename(
            columns={
                "absolute_shift": "lopo_absolute_shift",
                "direction_preserved_all": "lopo_direction_preserved_all",
            }
        ),
        on="candidate_pair",
        how="left",
    )
    .merge(
        bootstrap_summary[
            [
                "candidate_pair",
                "bootstrap_median",
                "bootstrap_ci_lower",
                "bootstrap_ci_upper",
                "direction_preservation_fraction",
            ]
        ].rename(
            columns={
                "direction_preservation_fraction": (
                    "bootstrap_direction_preservation_fraction"
                ),
            }
        ),
        on="candidate_pair",
        how="left",
    )
    .merge(
        continuous_confounder_robustness[
            [
                "candidate_pair",
                "most_influential_covariate",
                "worst_adjusted_median_correlation",
                "maximum_absolute_shift",
                "direction_preserved_all",
            ]
        ].rename(
            columns={
                "maximum_absolute_shift": "confounder_maximum_absolute_shift",
                "direction_preserved_all": (
                    "confounder_direction_preserved_all"
                ),
            }
        ),
        on="candidate_pair",
        how="left",
    )
    .merge(
        plate_robustness_summary[
            [
                "candidate_pair",
                "plate_adjusted_median",
                "plate_median_absolute_shift",
                "project_direction_preservation_fraction",
                "maximum_project_absolute_shift",
            ]
        ].rename(
            columns={
                "project_direction_preservation_fraction": (
                    "plate_project_direction_preservation_fraction"
                ),
            }
        ),
        on="candidate_pair",
        how="left",
    )
)

candidate_program_robustness_summary

,candidate_pair,rna_component,methylation_component,rna_seed_median,rna_subsample_median,methylation_seed_median,methylation_subsample_median,median_project_correlation,direction_consistency,minimum_nmf_concordance,...,bootstrap_ci_upper,bootstrap_direction_preservation_fraction,most_influential_covariate,worst_adjusted_median_correlation,confounder_maximum_absolute_shift,confounder_direction_preserved_all,plate_adjusted_median,plate_median_absolute_shift,plate_project_direction_preservation_fraction,maximum_project_absolute_shift
0,CROSS_OMIC_PAIR_02,RNA_IC083,METH_IC232,0.733854,0.817579,0.919843,0.871937,0.321344,0.90625,0.072895,...,0.403576,1.0,leukocyte_fraction,0.360627,0.039283,True,0.293377,0.075666,0.962963,0.298028
1,CROSS_OMIC_PAIR_03,RNA_IC184,METH_IC169,0.940041,0.950575,0.462809,0.538696,-0.179417,0.87500,0.103034,...,-0.130910,1.0,absolute_purity,-0.172493,0.006924,True,-0.169411,0.006108,1.000000,0.084541
2,CROSS_OMIC_PAIR_04,RNA_IC150,METH_IC128,0.989194,0.980672,0.972962,0.948443,0.133099,0.84375,0.316420,...,0.163422,1.0,leukocyte_fraction,0.076256,0.056843,True,0.123906,0.002370,0.962963,0.079880
3,CROSS_OMIC_PAIR_05,RNA_IC169,METH_IC023,0.393281,0.336027,0.368214,0.411761,-0.130233,0.84375,0.068668,...,-0.083505,1.0,gene_assigned_fraction_of_accounted,-0.122174,0.008059,True,-0.154498,0.017161,0.925926,0.048129
4,CROSS_OMIC_PAIR_06,RNA_IC175,METH_IC013,0.650641,0.571628,0.380165,0.406513,0.112888,0.78125,0.079366,...,0.140721,1.0,absolute_purity,0.094947,0.017940,True,0.112899,0.023180,1.000000,0.053613
5,CROSS_OMIC_PAIR_07,RNA_IC050,METH_IC234,0.494758,0.385271,0.295562,0.328067,0.110080,0.78125,0.073354,...,0.143072,1.0,leukocyte_fraction,0.118619,0.008539,True,0.117559,0.008284,0.962963,0.047096
6,CROSS_OMIC_PAIR_08,RNA_IC151,METH_IC050,0.623267,0.583573,0.312480,0.337208,0.109256,0.87500,0.089150,...,0.132338,1.0,external_panimmune_proliferation_score,0.100337,0.008919,True,0.101072,0.006776,1.000000,0.116117
7,CROSS_OMIC_PAIR_09,RNA_IC001,METH_IC107,0.448719,0.455537,0.337980,0.323272,0.107940,0.75000,0.066818,...,0.143753,1.0,leukocyte_fraction,0.138786,0.030845,True,0.112499,0.005649,0.962963,0.112194
8,CROSS_OMIC_PAIR_10,RNA_IC001,METH_IC241,0.448719,0.455537,0.369872,0.356044,0.107284,0.90625,0.048450,...,0.139160,1.0,absolute_purity,0.121324,0.014039,True,0.112588,0.004884,1.000000,0.073576
9,CROSS_OMIC_PAIR_11,RNA_IC193,METH_IC109,0.448745,0.381122,0.860798,0.851007,-0.105210,0.81250,0.175624,...,-0.066061,1.0,leukocyte_fraction,-0.088135,0.017075,True,-0.078703,0.024443,0.851852,0.131505


In [75]:
# =============================================================================
# Prepare final candidate-program robustness evidence table
# =============================================================================

candidate_program_robustness_evidence = (
    candidate_program_robustness_summary[
        [
            "candidate_pair",
            "rna_component",
            "methylation_component",
            "median_project_correlation",
            "direction_consistency",
            "lopo_absolute_shift",
            "lopo_direction_preserved_all",
            "bootstrap_ci_lower",
            "bootstrap_ci_upper",
            "bootstrap_direction_preservation_fraction",
            "confounder_maximum_absolute_shift",
            "confounder_direction_preserved_all",
            "plate_median_absolute_shift",
            "plate_project_direction_preservation_fraction",
            "rna_seed_median",
            "rna_subsample_median",
            "methylation_seed_median",
            "methylation_subsample_median",
            "minimum_nmf_concordance",
        ]
    ]
    .sort_values("candidate_pair")
    .reset_index(drop=True)
)

candidate_program_robustness_evidence

,candidate_pair,rna_component,methylation_component,median_project_correlation,direction_consistency,lopo_absolute_shift,lopo_direction_preserved_all,bootstrap_ci_lower,bootstrap_ci_upper,bootstrap_direction_preservation_fraction,confounder_maximum_absolute_shift,confounder_direction_preserved_all,plate_median_absolute_shift,plate_project_direction_preservation_fraction,rna_seed_median,rna_subsample_median,methylation_seed_median,methylation_subsample_median,minimum_nmf_concordance
0,CROSS_OMIC_PAIR_02,RNA_IC083,METH_IC232,0.321344,0.90625,0.025121,True,0.281203,0.403576,1.0,0.039283,True,0.075666,0.962963,0.733854,0.817579,0.919843,0.871937,0.072895
1,CROSS_OMIC_PAIR_03,RNA_IC184,METH_IC169,-0.179417,0.87500,0.002729,True,-0.221213,-0.130910,1.0,0.006924,True,0.006108,1.000000,0.940041,0.950575,0.462809,0.538696,0.103034
2,CROSS_OMIC_PAIR_04,RNA_IC150,METH_IC128,0.133099,0.84375,0.006823,True,0.082248,0.163422,1.0,0.056843,True,0.002370,0.962963,0.989194,0.980672,0.972962,0.948443,0.316420
3,CROSS_OMIC_PAIR_05,RNA_IC169,METH_IC023,-0.130233,0.84375,0.003629,True,-0.160662,-0.083505,1.0,0.008059,True,0.017161,0.925926,0.393281,0.336027,0.368214,0.411761,0.068668
4,CROSS_OMIC_PAIR_06,RNA_IC175,METH_IC013,0.112888,0.78125,0.002968,True,0.065783,0.140721,1.0,0.017940,True,0.023180,1.000000,0.650641,0.571628,0.380165,0.406513,0.079366
5,CROSS_OMIC_PAIR_07,RNA_IC050,METH_IC234,0.110080,0.78125,0.000805,True,0.069266,0.143072,1.0,0.008539,True,0.008284,0.962963,0.494758,0.385271,0.295562,0.328067,0.073354
6,CROSS_OMIC_PAIR_08,RNA_IC151,METH_IC050,0.109256,0.87500,0.000458,True,0.063231,0.132338,1.0,0.008919,True,0.006776,1.000000,0.623267,0.583573,0.312480,0.337208,0.089150
7,CROSS_OMIC_PAIR_09,RNA_IC001,METH_IC107,0.107940,0.75000,0.010208,True,0.059211,0.143753,1.0,0.030845,True,0.005649,0.962963,0.448719,0.455537,0.337980,0.323272,0.066818
8,CROSS_OMIC_PAIR_10,RNA_IC001,METH_IC241,0.107284,0.90625,0.005853,True,0.063589,0.139160,1.0,0.014039,True,0.004884,1.000000,0.448719,0.455537,0.369872,0.356044,0.048450
9,CROSS_OMIC_PAIR_11,RNA_IC193,METH_IC109,-0.105210,0.81250,0.002064,True,-0.135238,-0.066061,1.0,0.017075,True,0.024443,0.851852,0.448745,0.381122,0.860798,0.851007,0.175624


In [76]:
# =============================================================================
# Define program-robustness output paths
# =============================================================================

PROGRAM_ROBUSTNESS_DIR = CANDIDATE_CATALOG_PATH.parent

ROBUSTNESS_EVIDENCE_PATH = (
    PROGRAM_ROBUSTNESS_DIR
    / "tcga_cross_omic_candidate_program_robustness_evidence.csv"
)

DECOMPOSITION_STABILITY_PATH = (
    PROGRAM_ROBUSTNESS_DIR
    / "tcga_cross_omic_candidate_program_decomposition_stability.csv"
)

ROBUSTNESS_METADATA_PATH = (
    PROGRAM_ROBUSTNESS_DIR
    / "tcga_cross_omic_candidate_program_robustness_metadata.json"
)

In [77]:
# =============================================================================
# Prepare program-robustness metadata
# =============================================================================

program_robustness_metadata = {
    "analysis": "tcga_cross_omic_candidate_program_robustness",
    "candidate_program_count": len(candidate_program_robustness_evidence),
    "bootstrap": {
        "iterations": BOOTSTRAP_ITERATIONS,
        "random_seed": BOOTSTRAP_RANDOM_SEED,
        "sampling": "within_project",
    },
    "ica": {
        "random_seed": ICA_RANDOM_SEED,
        "algorithm": ICA_ALGORITHM,
        "whiten": ICA_WHITEN,
        "whiten_solver": ICA_WHITEN_SOLVER,
        "function": ICA_FUNCTION,
        "max_iter": ICA_MAX_ITER,
        "tolerance": ICA_TOLERANCE,
        "pca_components": ICA_PCA_COMPONENTS,
        "rna": {
            "components": RNA_ICA_COMPONENTS,
            "stability_seeds": list(RNA_ICA_STABILITY_SEEDS),
            "subsample_fraction": RNA_ICA_SUBSAMPLE_FRACTION,
            "subsample_seeds": list(RNA_ICA_SUBSAMPLE_SEEDS),
        },
        "methylation": {
            "components": METHYLATION_ICA_COMPONENTS,
            "stability_seeds": list(METHYLATION_ICA_STABILITY_SEEDS),
            "subsample_fraction": METHYLATION_ICA_SUBSAMPLE_FRACTION,
            "subsample_seeds": list(METHYLATION_ICA_SUBSAMPLE_SEEDS),
        },
        "subsampling_scope": (
            "Project-stratified sample subsampling conditional on the "
            "original PCA basis; not full end-to-end preprocessing stability."
        ),
    },
    "continuous_confounders": list(PRIMARY_CONTINUOUS_CONFOUNDERS),
    "alternative_purity_covariate": ALTERNATIVE_PURITY_COVARIATE,
    "evidence_policy": {
        "composite_robustness_score": False,
        "automatic_decomposition_stability_filter": False,
        "nmf_used_as_exclusion_criterion": False,
    },
    "interpretation_scope": (
        "Internal TCGA robustness assessment of candidate cross-omic "
        "programs. Robustness dimensions are retained separately and do "
        "not establish causal or clinical validity."
    ),
}

In [78]:
# =============================================================================
# Write program-robustness artifacts
# =============================================================================

candidate_program_robustness_evidence.to_csv(
    ROBUSTNESS_EVIDENCE_PATH,
    index=False,
)

candidate_pair_decomposition_stability.to_csv(
    DECOMPOSITION_STABILITY_PATH,
    index=False,
)

with open(ROBUSTNESS_METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(
        program_robustness_metadata,
        file,
        indent=2,
    )

print("Program-robustness artifacts written: 3")
print(f"Directory: {project_relative_path(PROGRAM_ROBUSTNESS_DIR)}")

Program-robustness artifacts written: 3
Directory: data/processed/tumor_programs


In [79]:
# =============================================================================
# Write detailed robustness-summary artifacts
# =============================================================================

detailed_robustness_outputs = {
    "tcga_cross_omic_candidate_program_lopo_summary.csv": lopo_summary,
    "tcga_cross_omic_candidate_program_bootstrap_summary.csv": bootstrap_summary,
    "tcga_cross_omic_candidate_program_confounder_summary.csv": (
        continuous_confounder_robustness
    ),
    "tcga_cross_omic_candidate_program_purity_sensitivity_summary.csv": (
        matched_purity_summary
    ),
    "tcga_cross_omic_candidate_program_plate_summary.csv": (
        plate_robustness_summary
    ),
}

for filename, table in detailed_robustness_outputs.items():
    table.to_csv(
        PROGRAM_ROBUSTNESS_DIR / filename,
        index=False,
    )

print(f"Detailed robustness summaries written: {len(detailed_robustness_outputs)}")

Detailed robustness summaries written: 5


In [80]:
# =============================================================================
# Finalize program-robustness artifact metadata
# =============================================================================

program_robustness_metadata["artifacts"] = {
    "robustness_evidence": ROBUSTNESS_EVIDENCE_PATH.name,
    "decomposition_stability": DECOMPOSITION_STABILITY_PATH.name,
    "detailed_summaries": list(detailed_robustness_outputs),
}

with open(ROBUSTNESS_METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(
        program_robustness_metadata,
        file,
        indent=2,
    )

## Summary

This notebook evaluated the robustness of the 13 cross-omic candidate programs retained from notebook 205 without repeating upstream discovery or QC.

### Main findings

- **Cross-project association stability:** all 13 candidate pairs preserved their association direction under leave-one-project-out analysis.
- **Resampling stability:** project-stratified bootstrap resampling preserved the association direction for all 13 pairs across all iterations, with empirical 95% intervals remaining on the expected side of zero.
- **Continuous-confounder sensitivity:** adjustment for tumor purity, leukocyte fraction, proliferation, RNA assignment quality, and methylation missingness preserved the cross-project association direction for every candidate.
- **Alternative purity sensitivity:** the matched ABSOLUTE-versus-CPE analysis supported the same qualitative association directions on the common evaluable project set.
- **Technical plate sensitivity:** median association direction remained preserved after plate adjustment, although sensitivity was heterogeneous across candidates. `CROSS_OMIC_PAIR_13` showed the clearest technical sensitivity, with marked attenuation after plate adjustment and reduced project-level direction preservation.
- **ICA representation stability:** the reference RNA and methylation ICA representations were reconstructed from the original notebook-205 inputs, with retained reference-component loadings reproduced exactly. Alternative ICA initializations and project-stratified sample subsampling revealed substantial heterogeneity in component stability across both modalities.
- **Cross-method sensitivity:** targeted NMF concordance from notebook 205 was retained as supporting evidence only and was not used as an exclusion criterion.

### Interpretation

The robustness analyses distinguish two separate properties:

1. **robustness of the cross-omic association across TCGA projects and sensitivity analyses**, and
2. **stability of the specific ICA representation used to describe each program**.

These properties are not interchangeable. Several candidate pairs retain stable cross-project associations despite sensitivity of one or both ICA components to decomposition perturbation.

Accordingly, no composite robustness score or post-hoc hard threshold was introduced. All 13 candidate programs are retained for downstream cross-system comparison, while their lineage, confounder, technical, and decomposition-sensitivity evidence is carried forward explicitly.

`CROSS_OMIC_PAIR_13` should be interpreted with particular caution in downstream analyses because of its plate sensitivity; this sensitivity is documented rather than used as an automatic exclusion rule.

### Outputs

Program-level robustness evidence, ICA decomposition-stability summaries, detailed leave-one-project-out, bootstrap, confounder, alternative-purity, and plate-sensitivity summaries, together with methodological provenance metadata, were written to:

`data/processed/tumor_programs`

### Scope

These results constitute an **internal TCGA robustness assessment** of candidate epigenetic-transcriptomic programs. They do not represent independent cross-dataset replication, causal validation, clinical prediction, or therapeutic evidence.

The tumor-discovery robustness layer is therefore complete and provides the TCGA-side candidate-program evidence required for subsequent independent comparison with cancer cell-model programs.